In [ ]:
import pandas as pd
import duckdb
import geopandas as gpd
import requests
import numpy as np
from shapely.geometry import Point
import plotly.graph_objects as go
import random
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import StandardScaler, MinMaxScaler,RobustScaler
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Patch
import matplotlib.cm as cm

import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [ ]:
import plotly.io as pio
pio.renderers.default = "browser"


## Fonction tracer la carte

In [ ]:
def carte_france(df, col_color, color_map, communes_fond, title="", labels_order=None):
    """
    df           : GeoDataFrame avec geometry et code_geo
    col_color    : colonne à colorier
    color_map    : dict {label: couleur_hex}
    communes_fond: GeoDataFrame de toutes les communes (fond gris)
    title        : titre de la carte
    labels_order : ordre des labels dans la légende (optionnel)
    """

    fig = plt.figure(figsize=(20, 10))

    # Métropole
    ax_metro = fig.add_axes([0.0, 0.0, 0.65, 1.0])
    communes_fond[~communes_fond['code_geo'].str.startswith('97')].plot(
        ax=ax_metro, color='lightgrey'
    )
    for label, color in color_map.items():
        df_zone = df[
            (df[col_color] == label) &
            (~df['code_geo'].str.startswith('97'))
        ]
        if len(df_zone) > 0:
            df_zone.plot(ax=ax_metro, color=color)

    ax_metro.set_title(title, fontsize=14)
    ax_metro.axis('off')

    # Légende
    ordre = labels_order if labels_order else list(color_map.keys())
    legend_elements = [Patch(facecolor=color_map[l], label=l) for l in ordre]
    ax_metro.legend(handles=legend_elements, loc='lower left', fontsize=8, title=col_color)

    # Inserts DOM-TOM
    inserts = {
        'Guadeloupe': ('971', [0.66, 0.75, 0.16, 0.22]),
        'Martinique': ('972', [0.83, 0.75, 0.16, 0.22]),
        'Guyane'    : ('973', [0.66, 0.50, 0.16, 0.22]),
        'Réunion'   : ('974', [0.83, 0.50, 0.16, 0.22]),
        'Mayotte'   : ('976', [0.66, 0.25, 0.16, 0.22]),
    }

    for nom, (dept, pos) in inserts.items():
        ax = fig.add_axes(pos)
        communes_fond[communes_fond['code_geo'].str.startswith(dept)].plot(
            ax=ax, color='lightgrey'
        )
        for label, color in color_map.items():
            df_zone = df[
                (df[col_color] == label) &
                (df['code_geo'].str.startswith(dept))
            ]
            if len(df_zone) > 0:
                df_zone.plot(ax=ax, color=color)
        ax.set_title(nom, fontsize=9)
        ax.axis('off')

    plt.show()

In [ ]:
def carte_france_continue(df, col_color, communes_fond, title="", cmap='YlOrRd'):


    # Couleurs
    vmin = df[col_color].quantile(0.02)
    vmax = df[col_color].quantile(0.98)
    norm = Normalize(vmin=vmin, vmax=vmax)
    colormap = cm.get_cmap(cmap)
    df = df.copy()
    df['_color'] = df[col_color].apply(
            lambda x: to_hex(colormap(norm(x))) if pd.notna(x) else '#d3d3d3'
        )

    inserts = {
        'Guadeloupe': ('971', [0.66, 0.75, 0.16, 0.22]),
        'Martinique': ('972', [0.83, 0.75, 0.16, 0.22]),
        'Guyane'    : ('973', [0.66, 0.50, 0.16, 0.22]),
        'Réunion'   : ('974', [0.83, 0.50, 0.16, 0.22]),
        'Mayotte'   : ('976', [0.66, 0.25, 0.16, 0.22]),
    }

    def plot_ax(ax, dept=None):
        if dept is None:
            fond = communes_fond[~communes_fond['code_geo'].str.startswith('97')]
            data = df[~df['code_geo'].str.startswith('97')]
        else:
            fond = communes_fond[communes_fond['code_geo'].str.startswith(dept)]
            data = df[df['code_geo'].str.startswith(dept)]

        fond.plot(ax=ax, color='lightgrey')
        if len(data) > 0:
            data.plot(ax=ax, color=list(data['_color']))
        ax.axis('off')

    fig = plt.figure(figsize=(20, 10))

    ax_metro = fig.add_axes([0.0, 0.0, 0.65, 1.0])
    plot_ax(ax_metro)
    ax_metro.set_title(title, fontsize=14)

    # Colorbar
    sm = ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax_metro, shrink=0.5, label=col_color)

    for nom, (dept, pos) in inserts.items():
        ax = fig.add_axes(pos)
        plot_ax(ax, dept=dept)
        ax.set_title(nom, fontsize=9)

    plt.show()

## Charger les données

Donnee de carto

In [ ]:

dept_geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-avec-outre-mer.geojson"
communes_geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/communes-avec-outre-mer.geojson"

# Télécharge d'abord avec requests
r = requests.get(dept_geojson_url, timeout=30)
with open("departements.geojson", "wb") as f:
    f.write(r.content)

dept_geo = gpd.read_file("departements.geojson")

r = requests.get(communes_geojson_url, timeout=30)
with open("communes.geojson", "wb") as f:
    f.write(r.content)

communes_geo = gpd.read_file("communes.geojson")

communes_geo["lat"] = communes_geo.geometry.centroid.y
communes_geo["lon"] = communes_geo.geometry.centroid.x

communes_geo.columns = ['code_geo', 'nom', 'geometry', 'lat', 'lon']

Donnees communes

In [ ]:
PCC_DUCKDB_FILE = "../dbt_pipeline/dev.duckdb"
con = duckdb.connect(database=PCC_DUCKDB_FILE, read_only=True)

In [ ]:


con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

rga_geo = con.sql("""
SELECT
    gid,
    niveau,
    surf_m2,
    ST_AsWKB(geom) AS geom,
    code_departement
FROM rga
""").df()

# convertir bytearray -> bytes
rga_geo["geom"] = gpd.GeoSeries.from_wkb(rga_geo["geom"].map(bytes))

rga_geo = gpd.GeoDataFrame(rga_geo, geometry="geom", crs="EPSG:2154")

# reprojection
rga_geo = rga_geo.to_crs("EPSG:4326")

In [ ]:
# Assure-toi que les deux sont en EPSG:4326
rga_geo = rga_geo.to_crs("EPSG:4326")
communes_geo = communes_geo.to_crs("EPSG:4326")

# Spatial join : quelle zone RGA intersecte chaque commune ?
communes_rga = gpd.sjoin(
    communes_geo[["code_geo", "nom", "geometry"]],
    rga_geo[["niveau", "geom"]],
    how="left",
    predicate="intersects"
)
communes_rga_max = communes_rga.groupby("code_geo")["niveau"].max().reset_index()
communes_rga_max.columns = ["code_geo", "niveau_rga_max"]

In [ ]:
azi_gaspar = con.sql("""SELECT	* FROM dev.main.azi_gaspar""").df()
ccr_details = con.sql("""SELECT	* FROM dev.main.ccr_details""").df()
catnat_gaspar = con.sql("""SELECT	* FROM dev.main.catnat_gaspar""").df()
geoportail_ccr_communes = con.sql("""SELECT	* FROM dev.main.geoportail_ccr_communes""").df()
pprn = con.sql("""SELECT	* FROM dev.main.pprn_gaspar""").df()

In [ ]:
colonnes = [
    "Point", "Latitude", "Longitude", "Niveau", "NORTMm_yr", "NORTMm_seas_JJA",
    "NORTMm_seas_DJF", "NORTXm_seas_JJA", "NORTX35D_yr", "NORTX30D_yr", "NORTR_yr",
    "NORRR_yr", "NORRR_seas_JJA", "NORRR_seas_DJF", "NORRRq99_yr", "NORRx1d_yr",
    "NORRRq99refD_yr", "NORIFM40_yr", "NORSWI04_yr", "ATMm_yr", "ATMm_seas_JJA",
    "ATMm_seas_DJF", "ATXm_seas_JJA", "ATX35D_yr", "ATX30D_yr", "ATR_yr",
    "ARRq99refD_yr", "AIFM40_yr", "ASWI04_yr", "ARRR_yr", "ARRR_seas_JJA",
    "ARRR_seas_DJF", "ARRRq99_yr", "ARRx1d_yr"
]
DRIAS = pd.read_csv(
    "../exploration/DRIAS.txt",
    sep=";",
    comment="#",
    names=colonnes,
    usecols=range(len(colonnes))  # ignore colonnes en trop
)

### Cat Nat

In [ ]:
mapping = {
    "Inondations et/ou Coulées de Boue": "inondation",
    "Inondations Remontée Nappe": "inondation",
    "Coulée de Boue": "inondation",
    "Lave Torrentielle": "inondation",

    "Sécheresse": "secheresse_rga",
    "Mouvement de Terrain": "secheresse_rga",

    "Glissement de Terrain": "mouvement_terrain",
    "Effondrement et/ou Affaisement": "mouvement_terrain",
    "Eboulement et/ou Chute de Blocs": "mouvement_terrain",
    "Glissement et Effondrement de Terrain": "mouvement_terrain",
    "Glissement et Eboulement Rocheux": "mouvement_terrain",

    "Tempête": "meteo",
    "Grêle": "meteo",
    "Poids de la Neige": "meteo",
    "Vents Cycloniques": "meteo",

    "Chocs Mécaniques liés à l'action des Vagues": "marin",
    "Raz de Marée": "marin",

    "Secousse Sismique": "sismique",
    "Eruption Volcanique": "sismique",

    "Divers": "autre"
}

ccr_details["type_catnat"] = ccr_details["nom_peril"].map(mapping).fillna("autre")

In [ ]:
ccr_details["annee"] = pd.to_datetime(ccr_details["date_debut_evenement"]).dt.year

In [ ]:
# Catégories principales
ccr_details["is_inondation"] = ccr_details["type_catnat"] == "inondation"
ccr_details["is_secheresse"] = ccr_details["type_catnat"] == "secheresse_rga"
ccr_details["is_mouvement_terrain"] = ccr_details["type_catnat"] == "mouvement_terrain"

# Arrêtés reconnus
ccr_details["is_reconnue"] = ccr_details["libelle_avis"] == "Reconnue"
ccr_details["is_reconnue_innondation"] = (ccr_details["libelle_avis"] == "Reconnue")&(ccr_details["is_inondation"])
ccr_details["is_reconnue_secheresse"] = (ccr_details["libelle_avis"] == "Reconnue")&(ccr_details["is_secheresse"] )

# Type de franchise
ccr_details["franchise_simple"] = ccr_details["franchise"] == "Simple"
ccr_details["franchise_double"] = ccr_details["franchise"] == "Doublée"
ccr_details["franchise_triple"] = ccr_details["franchise"] == "Triplée"
ccr_details["franchise_quadruple"] = ccr_details["franchise"] == "Quadruplée"

In [ ]:
stats_annee = (
    ccr_details.groupby(['code_geo', 'nom_commune',"annee"])
    .agg(
        nb_inondation=("is_inondation", "sum"),
        nb_secheresse=("is_secheresse", "sum"),
        nb_total=("code_arrete", "count"),
        nb_reconnue=("is_reconnue", "sum"),
        is_reconnue_innondation=("is_reconnue_innondation", "sum"),
        is_reconnue_secheresse=("is_reconnue_secheresse", "sum"),
        nb_franchise_simple=("franchise_simple", "sum"),
        nb_franchise_double=("franchise_double", "sum"),
        nb_franchise_triple=("franchise_triple", "sum"),
        nb_franchise_quadruple=("franchise_quadruple", "sum")
    )
    .reset_index()
)

## Donnee DRIAS

In [ ]:
variables_extremes = [
    'NORSWI04_yr', 'NORRR_yr', 'NORRRq99_yr',
    'NORTX35D_yr']
col = ['Latitude','Longitude']+variables_extremes

In [ ]:
geometry_communes = [Point(xy) for xy in zip(communes_geo['lon'], communes_geo['lat'])]
gdf_communes = gpd.GeoDataFrame(communes_geo, geometry=geometry_communes)
gdf_communes.set_crs(epsg=4326, inplace=True)

geometry_rga = [Point(xy) for xy in zip(DRIAS[col]['Longitude'], DRIAS[col]['Latitude'])]
gdf_stations = gpd.GeoDataFrame(DRIAS[col], geometry=geometry_rga)
gdf_stations.set_crs(epsg=4326, inplace=True)


def nearest_station(row, stations=gdf_stations):
    distances = stations.geometry.distance(row.geometry)
    nearest_idx = distances.idxmin()
    return stations.loc[nearest_idx]


variables_extremes = [
    'NORSWI04_yr', 'NORRR_yr', 'NORRRq99_yr',
    'NORTX35D_yr']

nearest_values = gdf_communes.apply(nearest_station, axis=1)

for var in variables_extremes:
    gdf_communes[var] = nearest_values[var].values


## Indice de Secheresse et Innondation

In [ ]:
Data_cat_nat_total = geoportail_ccr_communes.merge(communes_rga_max,on='code_geo')
Data_cat_nat_total = Data_cat_nat_total.merge(gdf_communes[['code_geo','NORSWI04_yr', 'NORRR_yr', 'NORRRq99_yr',
    'NORTX35D_yr']],left_on = 'code_geo',right_on= 'code_geo')
Data_cat_nat_total.loc[Data_cat_nat_total['code_geo'].isin(list(azi_gaspar['cod_commune'].unique())),'AZI']=1


catnat_cumul = (
    stats_annee
    .groupby("code_geo")[["nb_inondation", "nb_secheresse","nb_total"]]
    .sum()
    .reset_index()
)

Data_cat_nat_total = Data_cat_nat_total.merge(catnat_cumul,on='code_geo',how='outer')

In [ ]:


col_pca_secheresse = [
    'niveau_rga_max', 
    'NORSWI04_yr', 
    'NORTX35D_yr', 
    'nb_secheresse',
    'cout_sec_num_log'
]

# ,'NORRR_yr'
col_pca_inondation = ['AZI', 'NORRRq99_yr','nb_inondation','cout_ino_num_log']
col_prevention_sech = ['prevention_secheresse']  # 0 ou 1
col_prevention_ino = ['prevention_inondation']   # 0 ou 1
df_commune = Data_cat_nat_total.copy()
# tranche_map_ino = {
#     '≤ 100 k€'        : 50,
#     '100 - 500 k€'    : 150,
#     '500 - 1 000 k€'  : 750,
#     '1000 - 5 000 k€' : 2500,
#     '> 5 000 k€'      : 7500
# }
tranche_map_ino = {
    '≤ 2,5 k€'       : 1.25,
    '2,5 - 5 k€'    : 3.75,
     '5 - 10 k€'  : 7.5,
    '10 - 20 k€' : 15,
    '> 20 k€'     : 30
}

df_commune['cout_ino_num'] = df_commune['cout_moy_tout_ino'].map(tranche_map_ino)
df_commune['cout_sec_num'] = df_commune['cout_moy_sec'].map(tranche_map_ino)
df_commune['cout_ino_num_log'] = np.log1p(df_commune['cout_ino_num'])
df_commune['cout_sec_num_log'] = np.log1p(df_commune['cout_sec_num'])


df_commune[col_pca_secheresse] = df_commune[col_pca_secheresse].fillna(0)
df_commune[col_pca_inondation] = df_commune[col_pca_inondation].fillna(0)


scaler_sec = StandardScaler()
X_sec_scaled = scaler_sec.fit_transform(df_commune[col_pca_secheresse])

scaler_ino = StandardScaler()
X_ino_scaled = scaler_ino.fit_transform(df_commune[col_pca_inondation])


pca_sec = PCA(n_components=1)
df_commune['score_secheresse_pca'] = pca_sec.fit_transform(X_sec_scaled)

pca_ino = PCA(n_components=1)
df_commune['score_inondation_pca'] = pca_ino.fit_transform(X_ino_scaled)


scaler_global = MinMaxScaler()
df_commune[['score_secheresse_norm', 'score_inondation_norm']] = scaler_global.fit_transform(
    df_commune[['score_secheresse_pca','score_inondation_pca']]
)

# # Pondération prévention (0 = pas de prévention, 1 = prévention) 
# poids_prevention = 0.3  # ajustable
# df_commune['score_secheresse_net'] = df_commune['score_secheresse_norm'] - poids_prevention * df_commune['prevention_secheresse']
# df_commune['score_inondation_net'] = df_commune['score_inondation_norm'] - poids_prevention * df_commune['prevention_inondation']



df_commune['score_climatique_global'] = df_commune[['score_secheresse_norm','score_inondation_norm']].mean(axis=1)



In [ ]:

df_commune['score_secheresse_pca'] = pca_sec.transform(X_sec_scaled)
corr_sec = df_commune[col_pca_secheresse + ['score_secheresse_pca']].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr_sec, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Corrélation PCA Sécheresse vs variables")
plt.show()

In [ ]:

df_commune['score_inondation_pca'] = pca_ino.transform(X_ino_scaled)

corr_ino = df_commune[col_pca_inondation + ['score_inondation_pca']].corr()


plt.figure(figsize=(8,6))
sns.heatmap(corr_ino, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Corrélation PCA Innondation vs variables")
plt.show()

In [ ]:
df = df_commune[['code_geo','score_secheresse_norm','score_inondation_norm','score_climatique_global']]
df = df.merge(communes_geo,on='code_geo')

In [ ]:
col_color = 'score_secheresse_norm'
df = df.copy()
df_ = df[['code_geo','score_secheresse_norm']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df_ = df_.merge(communes_geo,on='code_geo')
df_ = gpd.GeoDataFrame(df_, geometry='geometry')


carte_france_continue(
    df=df_,
    col_color='score_secheresse_norm',
    communes_fond=communes_geo,
    title="Score Secheresse norm",
    cmap='YlOrRd'
)

In [ ]:
col_color = 'score_inondation_norm'
df = df.copy()
df_ = df[['code_geo','score_inondation_norm']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df_ = df_.merge(communes_geo,on='code_geo')
df_ = gpd.GeoDataFrame(df_, geometry='geometry')


carte_france_continue(
    df=df_,
    col_color='score_inondation_norm',
    communes_fond=communes_geo,
    title="Score Innondation norm",
    cmap='YlOrRd'
)

In [ ]:
col_color = 'score_climatique_global'
df = df.copy()
df_ = df[['code_geo','score_climatique_global']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df_ = df_.merge(communes_geo,on='code_geo')
df_ = gpd.GeoDataFrame(df_, geometry='geometry')


carte_france_continue(
    df=df_,
    col_color='score_climatique_global',
    communes_fond=communes_geo,
    title="Score Global",
    cmap='YlOrRd'
)

In [ ]:
df = gpd.GeoDataFrame(df, geometry='geometry')

fig = px.choropleth_map(
    df,
    geojson=df.geometry.__geo_interface__,
    locations=df.index,
    color="score_inondation_norm",
    color_continuous_scale=["white", "red"],
    hover_name="nom",
    map_style="carto-positron",
    zoom=4,
    center={"lat": 46.5, "lon": 2.5},
    opacity=0.6,
    title="Score Inondations",
)
fig.update_traces(marker_line_width=0) 
fig.show()

In [ ]:
df = gpd.GeoDataFrame(df, geometry='geometry')

fig = px.choropleth_map(
    df,
    geojson=df.geometry.__geo_interface__,
    locations=df.index,
    color="score_secheresse_norm",
    color_continuous_scale=["white", "red"],
    hover_name="nom",
    map_style="carto-positron",
    zoom=4,
    center={"lat": 46.5, "lon": 2.5},
    opacity=0.6,
    title="Score secheresse",
)
fig.update_traces(marker_line_width=0) 
fig.show()

## Indice de primes

### Charger les données

In [ ]:
primes_assurances_communes = con.sql("""SELECT	* FROM dev.main.primes_assurances_communes""").df()
budget_per_compte_communes = con.sql("""SELECT	* FROM dev.main.budget_per_compte_communes""").df()
budget_communes = con.sql("""SELECT	* FROM dev.main.budget_communes""").df()
geoportail_ccr_communes = con.sql("""SELECT	* FROM dev.main.geoportail_ccr_communes""").df()


In [ ]:
col = ['code_departement','siren', 'annee',
       'code_region','code_geo_from_siren', 'nom_com', 'solde_debiteur', 'solde_crediteur','solde']

primes_assurances_communes_group = primes_assurances_communes[col].groupby(by = ['code_departement','siren', 'annee',
       'code_region','code_geo_from_siren', 'nom_com']).sum().reset_index()

In [ ]:
primes_assurances_communes_group = primes_assurances_communes_group.rename(
    columns={'code_geo_from_siren': 'code_geo','solde':'primes_solde'}
)

In [ ]:
pivot_count = budget_per_compte_communes.pivot_table(
    index=['code_geo_from_siren','annee'],
    columns='type_compte',
    values='solde',  # n’importe quelle colonne, utilisée juste pour compter
    aggfunc='sum',
    fill_value=0
).reset_index()

In [ ]:
data_primes_budget = primes_assurances_communes_group.merge(pivot_count,left_on=['code_geo','annee'],right_on=['code_geo_from_siren','annee'])

### Traitements des primes

Fill pour toutes les années

In [ ]:
annees = range(data_primes_budget["annee"].min(), data_primes_budget["annee"].max() + 1)

# grille complète
index = pd.MultiIndex.from_product(
    [communes_geo['code_geo'], annees],
    names=["code_geo", "annee"]
)

primes_full = (
    data_primes_budget.set_index(["code_geo", "annee"])
      .reindex(index)
      .fillna(0)
      .reset_index()
)

In [ ]:
primes_full['prime_sur_produits'] = primes_full['primes_solde']/primes_full['depenses']
primes_full['prime_sur_depenses'] = primes_full['primes_solde']/primes_full['depenses']

In [ ]:
primes_full = primes_full.sort_values(['code_geo', 'annee'])

primes_full['gaussian'] = (
    primes_full.groupby('code_geo')['primes_solde']
    .transform(lambda x: gaussian_filter(x.values, sigma=1))
)

In [ ]:
primes_full["var_4ans"] = primes_full.groupby("code_geo")["gaussian"].transform(
    lambda x: x.pct_change(periods=4)
)
primes_full["var_10ans"] = primes_full.groupby("code_geo")["gaussian"].transform(
    lambda x: x.pct_change(periods=10)
)


In [ ]:

# Top 10 communes
last_year = primes_full["annee"].max()
df_last = primes_full[primes_full["annee"] == last_year]
df_last["var_4ans"] = df_last["var_4ans"]
top10 = df_last.sort_values("var_4ans", ascending=False).head(10)["code_geo"]


fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[str(code) for code in top10],
    specs=[[{"secondary_y": True}]*5,
           [{"secondary_y": True}]*5]
)

for idx, code in enumerate(top10):
    df_commune = primes_full[primes_full["code_geo"] == code]
    row = idx // 5 + 1
    col = idx % 5 + 1

    fig.add_trace(
        go.Scatter(x=df_commune["annee"], y=df_commune["gaussian"],
                   mode="lines+markers", name="Gaussian filter",
                   line=dict(color='blue', width=2)),
        row=row, col=col, secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=df_commune["annee"], y=df_commune["primes_solde"],
                   mode="lines+markers", name="Solde",
                   line=dict(color='green')),
        row=row, col=col, secondary_y=False
    )


    fig.add_trace(
        go.Scatter(x=df_commune["annee"], y=df_commune["prime_sur_depenses"],
                   mode="lines+markers", name="Dépense %",
                   line=dict(color='red', dash="dash")),
        row=row, col=col, secondary_y=True
    )


fig.update_layout(
    height=800, width=1500,
    title_text="Top 10 communes : solde/gaussian filter et dépenses %",
    template="plotly_white"
)

for i in range(1, 11):
    fig.update_yaxes(title_text="Solde / gaussian", row=(i-1)//5 + 1, col=(i-1)%5 + 1, secondary_y=False)
    fig.update_yaxes(title_text="Dépense %", row=(i-1)//5 + 1, col=(i-1)%5 + 1, secondary_y=True)

fig.show()

In [ ]:
primes_full["primes_solde"] = primes_full["primes_solde"].replace(0, pd.NA)
df_nozero = primes_full[primes_full["primes_solde"].notna()].copy()
df_nozero_count = df_nozero.groupby("code_geo")["annee"].count()
communes_ok = df_nozero_count[df_nozero_count >= 3].index
df_nozero_ok = df_nozero[df_nozero["code_geo"].isin(communes_ok)]

### Analyse différentes communes arretée des assurée et les communes qui viennent d'etre assurée apres 2020 ?

Communes assurée apres 2020

In [ ]:
primes_full["primes_solde"] = primes_full["primes_solde"].replace(0, pd.NA)
df_nozero = primes_full[primes_full["primes_solde"].notna()].copy()
df_nozero['annee_debut'] = df_nozero.groupby('code_geo')['annee'].transform('min')


df_nozero_count = df_nozero.groupby("code_geo")["annee"].count()
communes_ok = df_nozero_count[df_nozero_count >= 3].index
df_nozero_ok = df_nozero[df_nozero["code_geo"].isin(communes_ok)]


In [ ]:
# Première année d'assurance par commune
df_nozero['annee_debut'] = df_nozero.groupby('code_geo')['annee'].transform('min')

# Distribution des années de début
print(df_nozero.groupby('code_geo')['annee_debut'].first().value_counts().sort_index())

# Communes qui commencent après 2015 par exemple
nouvelles = df_nozero[df_nozero['annee_debut'] >= 2015]['code_geo'].nunique()
print(f"Communes nouvellement assurées : {nouvelles}")

In [ ]:

zero_2020 = primes_full[
    (primes_full['annee'] == 2020) & 
    (primes_full['primes_solde'].fillna(0) == 0)
]['code_geo'].nunique()

total_2020 = primes_full[primes_full['annee'] == 2020]['code_geo'].nunique()

print(f"Communes à 0 en 2020 : {zero_2020}")
print(f"Total communes en 2020 : {total_2020}")
print(f"Soit : {zero_2020/total_2020*100:.1f}%")

In [ ]:
codes_nouvelles = df_nozero[df_nozero['aa'] > 2020]['code_geo'].unique()

In [ ]:
codes_nouvelles = df_nozero[df_nozero['annee_debut'] > 2020]['code_geo'].unique()

In [ ]:
Communes_nouvelles = primes_full[primes_full['code_geo'].isin(codes_nouvelles)].copy()

Communes qui ont arrété d'etre assurées

In [ ]:

# Trier les données
primes_full = primes_full.sort_values(["code_geo", "annee"]).reset_index(drop=True)

codes_stop = []
liste_code_stop = []
for commune, df_c in primes_full.groupby("code_geo"):
    df_c = df_c.sort_values("annee").reset_index(drop=True)
    df_c["primes_solde"] = df_c["primes_solde"].fillna(0).values.astype(float)
    solde = df_c["primes_solde"].values
    years = df_c["annee"].values.astype(int)
    
    # Chercher la première année où le solde devient 0 après avoir été positif
    positive_years = np.where(solde > 0)[0]
    if len(positive_years)>0:
        zero_years_after = np.where(solde[positive_years[-1]+1:] == 0)[0]  # années après dernier positif
        
        if len(zero_years_after) > 0:
            stop_year = years[positive_years[-1] + 1 + zero_years_after[0]]
            codes_stop.append({
                "code_geo": commune,
                "annee_stop": stop_year
            })
            liste_code_stop.append(commune)


communes_stop_df = pd.DataFrame(codes_stop)

In [ ]:
df_communes_stop = primes_full.loc[primes_full['code_geo'].isin(list(communes_stop_df['code_geo'].unique()))]
df_communes_stop = df_communes_stop.loc[df_communes_stop['annee']==2024]
df_communes_stop = df_communes_stop.merge(communes_geo,on='code_geo')
df_communes_stop = gpd.GeoDataFrame(df_communes_stop, geometry='geometry')
df_communes_stop['categorie'] = 'Commune arrêtée'
fig = px.choropleth_map(
    df_communes_stop,
    geojson=df_communes_stop.geometry.__geo_interface__,
    locations=df_communes_stop.index,
    color='categorie',
    color_discrete_map={'Commune arrêtée': 'red'},
    hover_name="nom_com",
    map_style="carto-positron",
    zoom=4,
    center={"lat": 46.5, "lon": 2.5},
    opacity=0.6,
    title="Variation 4 years",
    range_color=[0,2]
)
fig.update_traces(marker_line_width=0) 
fig.show()

In [ ]:
fig = plt.figure(figsize=(20, 10))

# Métropole — grande carte à gauche
ax_metro = fig.add_axes([0.0, 0.0, 0.65, 1.0])
communes_geo_gdf[~communes_geo_gdf['code_geo'].str.startswith('97')].plot(
    ax=ax_metro, color='lightgrey', edgecolor='none'
)
df_communes_stop[~df_communes_stop['code_geo'].str.startswith('97')].plot(
    ax=ax_metro, color='red', edgecolor='none'
)
ax_metro.set_title("Communes ayant arrêté leur assurance", fontsize=14)
ax_metro.axis('off')

# Inserts DOM-TOM — petites cartes à droite
# [left, bottom, width, height] en fraction de la figure
inserts = {
    'Guadeloupe': ('971', [0.66, 0.75, 0.16, 0.22]),
    'Martinique': ('972', [0.83, 0.75, 0.16, 0.22]),
    'Guyane'    : ('973', [0.66, 0.50, 0.16, 0.22]),
    'Réunion'   : ('974', [0.83, 0.50, 0.16, 0.22]),
    'Mayotte'   : ('976', [0.66, 0.25, 0.16, 0.22]),
}

for nom, (dept, pos) in inserts.items():
    ax = fig.add_axes(pos)
    communes_geo_gdf[communes_geo_gdf['code_geo'].str.startswith(dept)].plot(
        ax=ax, color='lightgrey', edgecolor='none'
    )
    arretees = df_communes_stop[df_communes_stop['code_geo'].str.startswith(dept)]
    if len(arretees) > 0:
        arretees.plot(ax=ax, color='red', edgecolor='none')
    ax.set_title(nom, fontsize=9)
    ax.axis('off')

plt.show()

### Tracer Communes stop et communes nouvelles sur une carte

### variation sans ces communes

In [ ]:
codes_normales = primes_full[
    ~primes_full['code_geo'].isin(list(communes_stop_df['code_geo'].unique())) 
    # & 
    # ~primes_full['code_geo'].isin(codes_nouvelles)
]['code_geo'].unique()

In [ ]:
primes_full_complet = primes_full.loc[primes_full['code_geo'].isin(codes_normales)]

In [ ]:
# Regarder d'abord les percentiles extrêmes
var = primes_full_complet.groupby("code_geo")["var_4ans"].last()
print(var.describe(percentiles=[.01, .05, .25, .75, .95, .99]))

# Histogramme sans les outliers extrêmes (ex: entre 1% et 99%)
var_clipped = var.clip(
    var.quantile(0.01), 
    var.quantile(0.99)
)

plt.figure(figsize=(10, 6))
sns.histplot(var_clipped, bins=50, kde=True, color="skyblue")
plt.xlabel("Variation des primes sur 4 ans (sans outliers extrêmes)")
plt.ylabel("Nombre de communes")
plt.show()

In [ ]:
# # 1. Créer les groupes de variation
# var_last = primes_full_complet.groupby("code_geo")["var_4ans"].last().reset_index()
# var_last.columns = ["code_geo", "var_4ans_last"]

# bins = [-np.inf, 0, 0.5, 1.0, 2.0,3,4,5,6, np.inf]
# labels = ["diminue", "0-50%", "50-100%", "100-200%", "200-300%","300-400%","400-500%","500-600%",">600%"]

# var_last["groupe_var"] = pd.cut(var_last["var_4ans_last"], bins=bins, labels=labels)

# # 2. Merger sur primes_full_complet
# df_plot = primes_full_complet.merge(var_last[["code_geo", "groupe_var"]], on="code_geo", how="left")

# # 3. Tracer
# colors = sns.color_palette("RdYlGn_r", 10)

# plt.figure(figsize=(14, 7))

# for groupe, color in zip(labels, colors):
#     df_group = df_plot[df_plot["groupe_var"] == groupe].copy()
    
#     # Normaliser par valeur initiale
#     df_group["solde_norm"] = df_group.groupby("code_geo")["gaussian"].transform(
#         lambda x: (x - x.iloc[0]) / x.iloc[0] if x.iloc[0] != 0 else np.nan
#     )
    
#     group_mean = df_group.groupby("annee")["solde_norm"].mean()
#     group_std  = df_group.groupby("annee")["solde_norm"].std()
    
#     plt.plot(group_mean.index, group_mean.values, marker='o', color=color, linewidth=2, label=groupe)
#     plt.fill_between(group_mean.index, group_mean - group_std, group_mean + group_std, color=color, alpha=0.2)

# plt.xlabel("Année")
# plt.ylabel("Variation relative normalisée")
# plt.title("Évolution des primes par groupe de variation")
# plt.legend(title="Groupe", bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.tight_layout()
# plt.show()

In [ ]:

var_last = primes_full_complet.groupby("code_geo")["var_4ans"].last().reset_index()
var_last.columns = ["code_geo", "var_4ans_last"]

bins = [-np.inf, 0, 0.5, 1.0, 2.0,3,4,5,6, np.inf]
labels = ["diminue", "0-50%", "50-100%", "100-200%", "200-300%","300-400%","400-500%","500-600%",">600%"]

var_last["groupe_var"] = pd.cut(var_last["var_4ans_last"], bins=bins, labels=labels)


df_plot = primes_full_complet.merge(var_last[["code_geo", "groupe_var"]], on="code_geo", how="left")

colors = sns.color_palette("RdYlGn_r", 10)
plt.figure(figsize=(14, 7))

for groupe, color in zip(labels, colors):
    df_group = df_plot[df_plot["groupe_var"] == groupe].copy()
    
    # Normaliser par valeur initiale
    df_group["solde_norm"] = df_group.groupby("code_geo")["gaussian"].transform(
        lambda x: (x - x.iloc[0]) / x.iloc[0] if x.iloc[0] != 0 else np.nan
    )
    
    group_median = df_group.groupby("annee")["solde_norm"].median()
    group_p25    = df_group.groupby("annee")["solde_norm"].quantile(0.25)
    group_p75    = df_group.groupby("annee")["solde_norm"].quantile(0.75)

    plt.plot(
        group_median.index, group_median.values,
        marker='o', color=color, linewidth=2, label=groupe
    )
    plt.fill_between(
        group_median.index,
        group_p25,
        group_p75,
        color=color, alpha=0.2
    )

plt.xlabel("Année")
plt.ylabel("Variation relative normalisée")
plt.title("Évolution des primes par groupe de variation (bande = IQR)")
plt.legend(title="Groupe", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
bins = [-np.inf, 0, 0.5, 1.0, 2.0, 3, 4, 5, 6, np.inf]
labels = ["diminue", "0-50%", "50-100%", "100-200%", "200-300%", "300-400%", "400-500%", "500-600%", ">600%"]

colors_hex = [
    "#2ecc71",   # diminue      → vert
    "#a8d5a2",   # 0-50%        → vert clair
    "#f9e07f",   # 50-100%      → jaune
    "#f4a261",   # 100-200%     → orange doux
    "#e74c3c",   # 200-300%     → rouge vif 🔴
    "#c0392b",   # 300-400%     → rouge foncé
    "#8e44ad",   # 400-500%     → violet 🟣
    "#2c3e50",   # 500-600%     → noir bleuté ⚫
    "#000000",   # >600%        → noir
]

# colors_hex = [f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}" 
#               for r, g, b in sns.color_palette("RdYlGn_r", 9)]

color_map = dict(zip(labels, colors_hex))


df_2024 = df_plot.loc[df_plot['annee']==2024]
df_2024 = df_2024.merge(communes_geo,on='code_geo')
df_2024 = gpd.GeoDataFrame(df_2024, geometry='geometry')


fig = px.choropleth_map(
    df_2024,
    geojson=df_2024.geometry.__geo_interface__,
    locations=df_2024.index,
    color="groupe_var",
    color_discrete_map=color_map,
    category_orders={"groupe_var": labels},  # ordre de la légende
    hover_name="nom_com",
    map_style="carto-positron",
    zoom=4,
    center={"lat": 46.5, "lon": 2.5},
    opacity=0.6,
    title="Variation 4 years",
    range_color=[0,2]
)
fig.update_traces(marker_line_width=0) 
fig.show()

In [ ]:
carte_france(
    df=df_2024,
    col_color='groupe_var',
    color_map=color_map,
    communes_fond=communes_geo_gdf,
    title="Variation des primes sur 4 ans",
    labels_order=labels
)

In [ ]:
# # 1. Calculer les déciles sur var_4ans
# var_last["decile"] = pd.qcut(
#     var_last["var_4ans_last"],
#     q=10,
#     labels=False,  # 0 à 9
#     duplicates='drop'
# )

# # 2. Merger
# df_plot = primes_full_complet.merge(
#     var_last[["code_geo", "decile"]], 
#     on="code_geo", how="left"
# )

# # 3. Tracer
# colors = sns.color_palette("RdYlGn_r", 10)

# plt.figure(figsize=(14, 7))

# for decile_value, color in zip(sorted(df_plot['decile'].dropna().unique()), colors):
#     df_group = df_plot[df_plot["decile"] == decile_value].copy()
    
#     df_group["solde_norm"] = df_group.groupby("code_geo")["gaussian"].transform(
#         lambda x: (x - x.iloc[0]) / x.iloc[0] if x.iloc[0] != 0 else np.nan
#     )
    
#     group_median = df_group.groupby("annee")["solde_norm"].median()
#     group_p25    = df_group.groupby("annee")["solde_norm"].quantile(0.25)
#     group_p75    = df_group.groupby("annee")["solde_norm"].quantile(0.75)
    
#     plt.plot(group_median.index, group_median.values, 
#              marker='o', color=color, linewidth=2, label=f"D{decile_value+1}")
#     plt.fill_between(group_median.index, group_p25, group_p75, color=color, alpha=0.15)

# plt.xlabel("Année")
# plt.ylabel("Variation relative normalisée")
# plt.title("Évolution des primes par décile de variation")
# plt.legend(title="Décile", bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.tight_layout()
# plt.show()

In [ ]:
# decile_stats = var_last.groupby("decile")["var_4ans_last"].agg(['min', 'max', 'count']).round(3)
# decile_stats.index = [f"D{i+1}" for i in decile_stats.index]
# print(decile_stats)

## Etude de l'Acceleration des primes

In [ ]:
primes_full_complet['acceleration'] = (
    primes_full_complet['var_4ans'] - primes_full_complet['var_10ans']
)


seuil = primes_full_complet['acceleration'].quantile(0.75)
primes_full_complet['acceleration_forte'] = (
    primes_full_complet['acceleration'] > seuil
).astype(int)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    primes_full_complet['var_10ans'],
    primes_full_complet['var_4ans'],
    alpha=0.3, s=5
)
plt.xlim(-100, 1000)       # limite axe X
plt.ylim(-10, 200)  
plt.axline((0,0), slope=1, color='red', linestyle='--', label='var_4ans = var_10ans')
plt.xlabel('Variation 10 ans')
plt.ylabel('Variation 4 ans')
plt.title('Accélération des primes')
plt.legend()
plt.show()

## Etudes variation vs % depenses 

In [ ]:
primes_full_complet_2024 = primes_full_complet.loc[primes_full_complet['annee']==2024]

In [ ]:
import matplotlib.pyplot as plt
var_norm = primes_full_complet_2024['var_4ans'] / primes_full_complet_2024['var_4ans'].max()
prime_norm = primes_full_complet_2024['prime_sur_depenses'] / primes_full_complet_2024['prime_sur_depenses'].max()
colors = var_norm * prime_norm
primes_full_complet_2024['colors'] = colors
# Couleur = var_10ans, Taille = prime_sur_budget
plt.figure(figsize=(10,6))
sc = plt.scatter(
    primes_full_complet_2024['prime_sur_depenses'], 
    primes_full_complet_2024['var_4ans'], 
    c=colors,
    cmap='viridis',                  # palette
    alpha=0.7,
    vmin=0,               # valeur minimale pour la couleur
    vmax=0.00001,  
     s=2

)
plt.xlim(-0.1, 0.6)       # limite axe X
plt.ylim(-2, 10)  
plt.colorbar(sc, label="Variation sur 4 ans x Prime / depense 2024")
plt.xlabel("Prime / depense 2024")
plt.ylabel("Variation sur 4 ans ")
plt.title("Relation prime/depenses vs variation sur 4 ans")
plt.grid(True)
plt.show()

## Indices sur les primes

In [ ]:
cols = ['code_geo','nb_inondation', 'nb_secheresse', 'nb_total', 'nb_reconnue',
       'is_reconnue_innondation', 'is_reconnue_secheresse',
       'nb_franchise_simple', 'nb_franchise_double', 'nb_franchise_triple',
       'nb_franchise_quadruple']

In [ ]:
franchise  = stats_annee[cols].groupby(by='code_geo').sum()

In [ ]:
franchise['mean_franchise'] = (franchise['nb_franchise_simple']*1+franchise['nb_franchise_double']*2+franchise['nb_franchise_triple']*3+franchise['nb_franchise_quadruple']*4)/(franchise['nb_franchise_simple']+franchise['nb_franchise_double']+franchise['nb_franchise_triple']+franchise['nb_franchise_quadruple'])

In [ ]:
franchise = franchise.reset_index()

In [ ]:
stats_annee

In [ ]:
from matplotlib.colors import to_hex

In [ ]:
col_color = 'mean_franchise'
df = franchise.copy()
df = df[['code_geo','mean_franchise']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df = df.merge(communes_geo,on='code_geo')
df = gpd.GeoDataFrame(df, geometry='geometry')


carte_france_continue(
    df=df,
    col_color='mean_franchise',
    communes_fond=communes_geo,
    title="Franchise moyenne",
    cmap='YlOrRd'
)

In [ ]:
Assurance = primes_full_complet_2024.merge(franchise,on='code_geo',how='left')

In [ ]:
Assurance['mean_franchise'] = Assurance['mean_franchise'].fillna(1)

In [ ]:
# Regarder d'abord les percentiles extrêmes
var = primes_full_complet.groupby("code_geo")["var_4ans"].last()
print(var.describe(percentiles=[.01, .05, .25, .75, .95, .99]))

# Histogramme sans les outliers extrêmes (ex: entre 1% et 99%)
var_clipped = var.clip(
    var.quantile(0.01), 
    var.quantile(0.99)
)

plt.figure(figsize=(10, 6))
sns.histplot(var_clipped, bins=50, kde=True, color="skyblue")
plt.xlabel("Variation des primes sur 4 ans (sans outliers extrêmes)")
plt.ylabel("Nombre de communes")
plt.show()

In [ ]:
var_clipped = var_clipped.reset_index()
var_clipped.columns = ['code_geo','var_4ans_clip']

In [ ]:
Assurance = Assurance.merge(var_clipped, on ='code_geo')

In [ ]:
Assurance.columns

In [ ]:
scaler = MinMaxScaler()
Assurance[['var_4ans_clip', 'prime_depenses_norm', 'franchise_norm']] = scaler.fit_transform(
    Assurance[['var_4ans_clip', 'prime_sur_depenses', 'mean_franchise']]
)

# Combinaison pondérée — à ajuster selon ton expertise métier
Assurance['score_assurance_final'] = (
    0.4 * Assurance['var_4ans_clip'] +
    0.4 * Assurance['prime_depenses_norm'] +
    0.2 * Assurance['franchise_norm']  # moins de poids car peu de variance
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['var_4ans_clip', 'prime_depenses_norm', 'franchise_norm']):
    Assurance[col].hist(bins=50, ax=ax, edgecolor='white', color='steelblue')
    ax.axvline(Assurance[col].mean(), color='red', linestyle='--', label=f'moy={Assurance[col].mean():.2f}')
    ax.axvline(Assurance[col].median(), color='orange', linestyle='--', label=f'med={Assurance[col].median():.2f}')
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(Assurance[['var_4ans_clip', 'prime_depenses_norm', 'franchise_norm']].describe(
    percentiles=[.01, .05, .25, .75, .95, .99]
))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['var_4ans_clip', 'prime_depenses_norm', 'mean_franchise']):
    Assurance[col].hist(bins=50, ax=ax, edgecolor='white', color='steelblue')
    ax.axvline(Assurance[col].mean(), color='red', linestyle='--', label=f'moy={Assurance[col].mean():.2f}')
    ax.axvline(Assurance[col].median(), color='orange', linestyle='--', label=f'med={Assurance[col].median():.2f}')
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(Assurance[['var_4ans_clip', 'prime_depenses_norm', 'mean_franchise']].describe(
    percentiles=[.01, .05, .25, .75, .95, .99]
))

In [ ]:
Assurance['score_assurance_final'] = (
    0.5 * Assurance['var_4ans_clip'] +
    0.5 * Assurance['prime_depenses_norm']
)

# Bonus si franchise modulée
Assurance['score_assurance_final'] += (
     (Assurance['mean_franchise']-1)
).clip(0, 0.2)

In [ ]:
col_color = 'mean_franchise'
df = Assurance.copy()
df = df[['code_geo','score_assurance_final']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df = df.merge(communes_geo,on='code_geo')
df = gpd.GeoDataFrame(df, geometry='geometry')


carte_france_continue(
    df=df,
    col_color='score_assurance_final',
    communes_fond=communes_geo,
    title="Score Assurance",
    cmap='YlOrRd'
)

In [ ]:
col_color = 'prime_depenses_norm'
df = Assurance.copy()
df = df[['code_geo','prime_depenses_norm']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df = df.merge(communes_geo,on='code_geo')
df = gpd.GeoDataFrame(df, geometry='geometry')


carte_france_continue(
    df=df,
    col_color='prime_depenses_norm',
    communes_fond=communes_geo,
    title="Franchise moyenne",
    cmap='YlOrRd'
)

### Budget

In [ ]:
population_code_geo = con.sql("""SELECT	* FROM dev.main.population_code_geo""").df()

In [ ]:
df_budget  = data_primes_budget.merge(population_code_geo,on='code_geo')

In [ ]:
df_budget.columns

In [ ]:
def get_pop(row):
    annee = row['annee']
    if annee <= 2022:
        return row['pop_2022']
    elif annee == 2023:
        return row['pop_2023']
    else:  # 2024+
        return row['pop_2024']

df_budget['population'] = df_budget.apply(get_pop, axis=1)
df_budget['population'] = df_budget['population'].replace(0, np.nan)
df_budget['solde'] = df_budget['produits']  - df_budget['depenses'] 

df_budget['taux_endettement'] = df_budget['dettes'] / df_budget['produits'].replace(0, np.nan)


# 2. Calcul par habitant
df_budget['depenses_hab'] = df_budget['depenses'] / df_budget['population'].replace(0, np.nan)
df_budget['dettes_hab'] = df_budget['dettes'] / df_budget['population'].replace(0, np.nan)
df_budget['produits_hab'] = df_budget['produits'] / df_budget['population'].replace(0, np.nan)
df_budget['solde_hab'] = df_budget['solde'] / df_budget['population'].replace(0, np.nan)

# 3. Moyenne sur plusieurs années par commune
df_moy = df_budget.groupby('code_geo')[
    ['depenses_hab', 'dettes_hab', 'produits_hab', 'solde_hab','depenses','dettes','produits','solde','taux_endettement','cap_autofinancement']
].mean().reset_index()

In [ ]:
data_primes_budget[data_primes_budget['code_geo'].str.startswith('976')].shape


In [ ]:
# Vérifier Mayotte
print(df_budget[df_budget['code_geo'].str.startswith('97')].shape)
print(df_budget[df_budget['code_geo'].str.startswith('97')])

In [ ]:


cols_socio = ['produits_hab', 'depenses_hab', 'solde_hab', 
              'taux_endettement']

plt.figure(figsize=(8, 6))
sns.heatmap(
    df_moy[cols_socio].corr(), 
    annot=True, 
    fmt=".2f", 
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1
)
plt.title("Corrélations variables socio-économiques")
plt.tight_layout()
plt.show()

In [ ]:
df_moy[cols_socio].corr()

In [ ]:
# Clip d'abord
for col in ['solde_hab', 'taux_endettement']:
    df_moy[col + '_clip'] = df_moy[col].clip(
        df_moy[col].quantile(0.01),
        df_moy[col].quantile(0.99)
    )

# Visualiser après clip
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['solde_hab_clip', 'taux_endettement_clip']):
    df_moy[col].hist(bins=50, ax=ax, edgecolor='white', color='steelblue')
    ax.axvline(df_moy[col].mean(), color='red', linestyle='--', label=f'moy={df_moy[col].mean():.2f}')
    ax.axvline(df_moy[col].median(), color='orange', linestyle='--', label=f'med={df_moy[col].median():.2f}')
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(df_moy[['produits_hab_clip', 'taux_endettement_clip']].describe(
    percentiles=[.01, .05, .25, .75, .95, .99]
))

In [ ]:
scaler = MinMaxScaler()
df_moy[['solde_hab_norm', 'endettement_norm']] = scaler.fit_transform(
    df_moy[['solde_hab_clip', 'taux_endettement_clip']]
)

df_moy['solde_hab_inv'] = 1 - df_moy['solde_hab_norm']

df_moy['score_socio_eco'] = (
    0.5 * df_moy['solde_hab_inv'] +
    0.5 * df_moy['endettement_norm']
)

print(df_moy['score_socio_eco'].describe(percentiles=[.25, .5, .75, .9, .95]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution des 2 composantes
for ax, col, color in zip(axes[:2], ['solde_hab_inv', 'endettement_norm'], ['steelblue', 'orange']):
    df_moy[col].hist(bins=50, ax=ax, edgecolor='white', color=color)
    ax.axvline(df_moy[col].mean(), color='red', linestyle='--', label=f'moy={df_moy[col].mean():.2f}')
    ax.axvline(df_moy[col].median(), color='black', linestyle='--', label=f'med={df_moy[col].median():.2f}')
    ax.set_title(col)
    ax.legend(fontsize=8)

# Distribution du score final
df_moy['score_socio_eco'].hist(bins=50, ax=axes[2], edgecolor='white', color='green')
axes[2].axvline(df_moy['score_socio_eco'].mean(), color='red', linestyle='--', label=f'moy={df_moy["score_socio_eco"].mean():.2f}')
axes[2].axvline(df_moy['score_socio_eco'].median(), color='black', linestyle='--', label=f'med={df_moy["score_socio_eco"].median():.2f}')
axes[2].set_title('score_socio_eco')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(df_moy['score_socio_eco'].describe(percentiles=[.25, .5, .75, .9, .95]))

In [ ]:
col_color = 'solde_hab_inv'
df_moy = df_moy.copy()
df_moy_ = df_moy[['code_geo','solde_hab_inv']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df_moy_ = df_moy_.merge(communes_geo,on='code_geo')
df_moy_ = gpd.GeoDataFrame(df_moy_, geometry='geometry')


carte_france_continue(
    df=df_moy_,
    col_color='solde_hab_inv',
    communes_fond=communes_geo,
    title="Solde par habitant",
    cmap='YlOrRd'
)

In [ ]:
col_color = 'endettement_norm'
df_moy = df_moy.copy()
df_moy_ = df_moy[['code_geo','endettement_norm']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df_moy_ = df_moy_.merge(communes_geo,on='code_geo')
df_moy_ = gpd.GeoDataFrame(df_moy_, geometry='geometry')


carte_france_continue(
    df=df_moy_,
    col_color='endettement_norm',
    communes_fond=communes_geo,
    title="Franchise moyenne",
    cmap='YlOrRd'
)

In [ ]:
col_color = 'score_socio_eco'
df_moy = df_moy.copy()
df_moy_ = df_moy[['code_geo','score_socio_eco']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df_moy_ = df_moy_.merge(communes_geo,on='code_geo')
df_moy_ = gpd.GeoDataFrame(df_moy_, geometry='geometry')


carte_france_continue(
    df=df_moy_,
    col_color='score_socio_eco',
    communes_fond=communes_geo,
    title="Score socio eco",
    cmap='YlOrRd'
)

In [ ]:
df_final = df_moy.merge(Assurance,on='code_geo')
df_final = df_final.merge(df_commune,on='code_geo')


In [ ]:
cols_scores = ['score_assurance_final', 'score_secheresse_norm', 
               'score_inondation_norm', 'score_socio_eco']

print(df_final[cols_scores].describe().loc[['min', 'max', 'mean', 'std']])

In [ ]:
# Poids selon ton classement — à ajuster selon ton expertise métier
w_assurance  = 0.40
w_secheresse = 0.2
w_inondation = 0.2
w_socio      = 0.2

assert w_assurance + w_secheresse + w_inondation + w_socio == 1.0

df_final['score_global'] = (
    w_assurance  * df_final['score_assurance_final'] +
    w_secheresse * df_final['score_secheresse_norm'] +
    w_inondation * df_final['score_inondation_norm'] +
    w_socio      * df_final['score_socio_eco']
)

print(df_final['score_global'].describe(percentiles=[.25, .5, .75, .9, .95]))

In [ ]:
col_color = 'score_global'
df_final = df_final.copy()
df_ = df_final[['code_geo','score_global']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df_ = df_.merge(communes_geo,on='code_geo')
df_ = gpd.GeoDataFrame(df_, geometry='geometry')


carte_france_continue(
    df=df_,
    col_color='score_global',
    communes_fond=communes_geo,
    title="Franchise moyenne",
    cmap='YlOrRd'
)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Préparer les données
cols_scores = ['score_assurance_final', 'score_secheresse_norm', 
               'score_inondation_norm', 'score_socio_eco']

X = df_final[cols_scores].fillna(0)

# 2. Méthode du coude pour choisir k
inerties = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inerties.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(2, 11), inerties, 'bo-')
plt.xlabel('Nombre de clusters')
plt.ylabel('Inertie')
plt.title('Méthode du coude')
plt.show()

# 3. KMeans avec 5 clusters
km = KMeans(n_clusters=5, random_state=42, n_init=10)
df_final['cluster'] = km.fit_predict(X)

# 4. Profil moyen de chaque cluster
profils = df_final.groupby('cluster')[cols_scores + ['score_global']].mean().round(3)
print(profils.sort_values('score_global'))

In [ ]:
df_final['score_climatique'] = df_final[['score_secheresse_norm', 'score_inondation_norm']].max(axis=1)

In [ ]:
# Seuils au 75e percentile
cols_scores = ['score_assurance_final', 'score_climatique', 'score_socio_eco']

seuils = {col: df_final[col].quantile(0.75) for col in cols_scores}

# Flags
for col, seuil in seuils.items():
    df_final[col + '_flag'] = (df_final[col] > seuil).astype(int)

# Nombre de risques élevés
df_final['nb_risques_eleves'] = (
    df_final['score_assurance_final_flag'] +
    df_final['score_climatique_flag'] +
    df_final['score_socio_eco_flag']
)

# Distribution
print(df_final['nb_risques_eleves'].value_counts().sort_index())

# Communes qui cumulent les 3
df_critique = df_final[df_final['nb_risques_eleves'] == 3]
print(f"\nCommunes critiques (3 risques élevés) : {len(df_critique)}")
print(df_critique[['code_geo'] + cols_scores + ['score_global']].sort_values('score_global', ascending=False).head(20))

In [ ]:
color_map_risques = {
    0: '#2ecc71',   # vert — peu vulnérable
    1: '#f1c40f',   # jaune — 1 risque élevé
    2: '#e67e22',   # orange — 2 risques élevés
    3: '#e74c3c',   # rouge — 3 risques élevés ← prioritaires
}
df_ = df_final[['code_geo','nb_risques_eleves']]
communes_geo = gpd.GeoDataFrame(communes_geo, geometry='geometry')
df_ = df_.merge(communes_geo,on='code_geo')
df_ = gpd.GeoDataFrame(df_, geometry='geometry')


df_['nb_risques_eleves_str'] = df_['nb_risques_eleves'].map({
    0: '0 risque',
    1: '1 risque',
    2: '2 risques',
    3: '3 risques'
})
carte_france(
    df=df_,
    col_color='nb_risques_eleves_str',
    color_map={
        '0 risque' : '#2ecc71',
        '1 risque' : '#f1c40f',
        '2 risques': '#e67e22',
        '3 risques': '#A11C03',
    },
    communes_fond=communes_geo,
    title="Cumul des risques par commune",
    labels_order=['0 risque', '1 risque', '2 risques', '3 risques']
)